# Anhedonic AI — Full Experiment Analysis

**Part 1**: Steering Vector Discovery — finding the reward (liking) direction in Qwen2VL

**Part 2**: Multi-Run Behavioral Evaluation — 200 tasks × 5 runs with proper error bars

---

# Part 1: Steering Vector Discovery

## Block 1: Imports

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from scipy import stats

# Style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.family'] = 'sans-serif'

C_NORMAL = '#4A90D9'
C_ANHEDONIC = '#E85D4A'

print('All imports OK')

## Block 2: Load Activations

In [ ]:
high_reward_activations = torch.load('high_reward_activations.pt')
neutral_activations = torch.load('neutral_activations.pt')
steering_vectors = torch.load('steering_vectors.pt')

num_layers = len(high_reward_activations[0])
num_pairs = len(high_reward_activations)

print(f'Layers: {num_layers}')
print(f'Contrastive sentence pairs: {num_pairs}')

## Block 3: Separation Score Per Layer

For each layer, we compute the L2 distance between the mean reward activation and the mean neutral activation. This shows where the reward signal builds up.

In [ ]:
separation_scores = []
for layer_idx in range(num_layers):
    high_stack = torch.stack([a[layer_idx].squeeze() for a in high_reward_activations])
    neutral_stack = torch.stack([a[layer_idx].squeeze() for a in neutral_activations])
    score = (high_stack.mean(0) - neutral_stack.mean(0)).norm().item()
    separation_scores.append(score)

plt.figure(figsize=(14, 5))
colors = ['tomato' if s == max(separation_scores) else 'steelblue' for s in separation_scores]
plt.bar(range(num_layers), separation_scores, color=colors)
plt.xlabel('Layer')
plt.ylabel('Separation Score (L2 norm)')
plt.title('Reward Signal Strength Per Layer')
plt.xticks(range(num_layers))
plt.axvline(x=np.argmax(separation_scores), color='red', linestyle='--', alpha=0.5,
            label=f'Best layer: {np.argmax(separation_scores)}')
plt.axvspan(6, 27, alpha=0.08, color='red', label='Intervention range')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Best layer: {np.argmax(separation_scores)} (score: {max(separation_scores):.2f})')

## Block 4: PCA Visualization

In [ ]:
best_layer = int(np.argmax(separation_scores))

for target_layer in [20, 24, best_layer]:
    high_stack = torch.stack([a[target_layer].squeeze() for a in high_reward_activations]).float().numpy()
    neutral_stack = torch.stack([a[target_layer].squeeze() for a in neutral_activations]).float().numpy()

    all_acts = np.concatenate([high_stack, neutral_stack], axis=0)
    labels = ['Reward'] * num_pairs + ['Neutral'] * num_pairs

    pca = PCA(n_components=2)
    reduced = pca.fit_transform(all_acts)

    plt.figure(figsize=(7, 5))
    for i, (x, y) in enumerate(reduced):
        color = 'tomato' if labels[i] == 'Reward' else 'steelblue'
        plt.scatter(x, y, color=color, s=80, alpha=0.8)

    plt.scatter([], [], color='tomato', label='High Reward')
    plt.scatter([], [], color='steelblue', label='Neutral')
    plt.legend(fontsize=12)
    plt.title(f'PCA at Layer {target_layer}')
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    plt.tight_layout()
    plt.show()

## Block 5: Linear Probe with Cross-Validation

PCA finds maximum variance, not maximum separation. A logistic regression probe finds the direction that best separates the two conditions.

In [ ]:
for target_layer in [20, 24, 27]:
    high_stack = torch.stack([a[target_layer].squeeze() for a in high_reward_activations]).float().numpy()
    neutral_stack = torch.stack([a[target_layer].squeeze() for a in neutral_activations]).float().numpy()

    X = np.concatenate([high_stack, neutral_stack], axis=0)
    y = np.array([1] * len(high_stack) + [0] * len(neutral_stack))

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_scaled, y)
    acc = accuracy_score(y, clf.predict(X_scaled))
    cv = cross_val_score(clf, X_scaled, y, cv=5, scoring='accuracy')

    direction = clf.coef_[0]
    projections = X_scaled @ direction

    plt.figure(figsize=(10, 4))
    plt.hist(projections[y == 1], bins=30, alpha=0.6, color='tomato', label='High Reward')
    plt.hist(projections[y == 0], bins=30, alpha=0.6, color='steelblue', label='Neutral')
    plt.title(f'Layer {target_layer} \u2014 Train: {acc*100:.1f}% | 5-fold CV: {cv.mean()*100:.1f}% (\u00b1{cv.std()*100:.1f}%)')
    plt.xlabel('Projection onto reward direction')
    plt.ylabel('Count')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Block 6: Steering Vector Magnitude

In [ ]:
magnitudes = [steering_vectors[i].norm().item() for i in range(num_layers)]

plt.figure(figsize=(14, 4))
plt.plot(range(num_layers), magnitudes, marker='o', color='darkorange', linewidth=2)
plt.fill_between(range(num_layers), magnitudes, alpha=0.2, color='darkorange')
plt.xlabel('Layer')
plt.ylabel('Steering Vector Magnitude')
plt.title('Reward Signal Build-Up Across Layers')
plt.xticks(range(num_layers))
plt.axvspan(6, 27, alpha=0.1, color='red', label='Intervention range (layers 6-27)')
plt.legend()
plt.tight_layout()
plt.show()

## Block 7: Per-Pair Distance

In [ ]:
df_contrastive = pd.read_csv('contrastive_dataset.csv')

distances = []
for i in range(num_pairs):
    h = high_reward_activations[i][best_layer].squeeze().float()
    n = neutral_activations[i][best_layer].squeeze().float()
    distances.append((h - n).norm().item())

plt.figure(figsize=(14, 5))
colors = ['tomato' if d > np.mean(distances) else 'steelblue' for d in distances]
plt.bar(range(num_pairs), distances, color=colors)
plt.axhline(y=np.mean(distances), color='black', linestyle='--', label=f'Mean: {np.mean(distances):.2f}')
plt.xlabel('Sentence Pair Index')
plt.ylabel('Distance (L2)')
plt.title(f'Per-Pair Separation at Layer {best_layer}')
plt.legend()
plt.tight_layout()
plt.show()

top5 = np.argsort(distances)[::-1][:5]
print('Top 5 most separated pairs:')
for idx in top5:
    print(f'  Pair {idx}: {distances[idx]:.2f}')
    print(f'    Reward:  {df_contrastive.iloc[idx]["high_reward"]}')
    print(f'    Neutral: {df_contrastive.iloc[idx]["neutral"]}')
    print()

---
---

# Part 2: Multi-Run Behavioral Evaluation

**Design**: 200 tasks (5 types \u00d7 40) \u00d7 2 conditions \u00d7 5 runs = 2,000 generations

**Intervention**: Liking probe projected out at layers 6-27, \u03b1=1.0

**Sampling**: temperature=0.7, top_p=0.9, different seed per run

**Statistics**: Paired t-test across 5 runs (each run is one observation)

## Block 8: Load Multi-Run Results

In [ ]:
df_raw = pd.read_csv('eval_multirun_raw.csv')
df_summary = pd.read_csv('eval_multirun_summary.csv')

NUM_RUNS = len(df_summary)
tasks_per_run = len(df_raw) // NUM_RUNS

print(f'Runs: {NUM_RUNS}')
print(f'Tasks per run: {tasks_per_run}')
print(f'Total responses: {len(df_raw)}')
print(f'\nTask types per run:')
print(df_raw[df_raw['run_id']==0]['task_type'].value_counts().to_string())

## Block 9: Overall Positive Emotion Words (Primary Metric)

This is our primary metric: does the anhedonic model use fewer positive emotion words overall?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

n_vals = df_summary['normal_pos_words'].values
a_vals = df_summary['anhedonic_pos_words'].values

n_mean, a_mean = n_vals.mean(), a_vals.mean()
n_sem = n_vals.std() / np.sqrt(NUM_RUNS)
a_sem = a_vals.std() / np.sqrt(NUM_RUNS)

bars = ax.bar(['Normal', 'Anhedonic'], [n_mean, a_mean],
              yerr=[n_sem, a_sem], color=[C_NORMAL, C_ANHEDONIC],
              capsize=10, width=0.5, edgecolor='black', linewidth=0.5)

# Individual run data points
jitter = 0.05
for i, (nv, av) in enumerate(zip(n_vals, a_vals)):
    ax.plot(0 + np.random.uniform(-jitter, jitter), nv, 'o',
            color='black', alpha=0.5, markersize=6, zorder=5)
    ax.plot(1 + np.random.uniform(-jitter, jitter), av, 'o',
            color='black', alpha=0.5, markersize=6, zorder=5)
    # Connect paired runs
    ax.plot([0, 1], [nv, av], '-', color='gray', alpha=0.3, linewidth=1)

# Stats
t_stat, p_val = stats.ttest_rel(n_vals, a_vals, alternative='greater')
pct_drop = (1 - a_mean / n_mean) * 100
sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'n.s.'

ymax = max(n_mean + n_sem, a_mean + a_sem)
ax.plot([0, 0, 1, 1], [ymax*1.05, ymax*1.08, ymax*1.08, ymax*1.05], 'k-', linewidth=1.2)
ax.text(0.5, ymax*1.09, f'p={p_val:.4f} {sig}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Avg Positive Emotion Words per Response', fontsize=12)
ax.set_title(f'Overall Hedonic Tone ({pct_drop:.0f}% reduction)', fontsize=14, fontweight='bold')
ax.set_ylim(0, ymax * 1.2)

plt.tight_layout()
plt.show()

print(f'Normal:    {n_mean:.3f} \u00b1 {n_vals.std():.3f} (mean \u00b1 SD across {NUM_RUNS} runs)')
print(f'Anhedonic: {a_mean:.3f} \u00b1 {a_vals.std():.3f}')
print(f'Paired t-test: t={t_stat:.3f}, p={p_val:.6f} {sig}')

## Block 10: Per-Task-Type Breakdown

Which task types show the strongest effect? This reveals the specificity of our intervention.

In [ ]:
task_types = ['preference_ranking', 'scenario_continuation', 'reward_vs_neutral',
              'anticipation', 'effort_willingness']
task_labels = ['Preference\nRanking', 'Scenario\nContinuation', 'Reward vs\nNeutral',
               'Anticipation', 'Effort\nWillingness\n(control)']

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(task_types))
width = 0.32

n_means, a_means, n_sems, a_sems, p_vals, pct_drops = [], [], [], [], [], []

for tt in task_types:
    n_col = f'{tt}_normal_pos'
    a_col = f'{tt}_anhedonic_pos'
    n_v = df_summary[n_col].values
    a_v = df_summary[a_col].values
    n_means.append(n_v.mean())
    a_means.append(a_v.mean())
    n_sems.append(n_v.std() / np.sqrt(NUM_RUNS))
    a_sems.append(a_v.std() / np.sqrt(NUM_RUNS))
    
    if n_v.std() == 0 and a_v.std() == 0:
        if np.allclose(n_v, a_v):
            p_vals.append(float('nan'))
        else:
            p_vals.append(0.0)
    else:
        try:
            t, p = stats.ttest_rel(n_v, a_v, alternative='greater')
            p_vals.append(p)
        except:
            p_vals.append(float('nan'))
    
    if n_v.mean() > 0:
        pct_drops.append((1 - a_v.mean() / n_v.mean()) * 100)
    else:
        pct_drops.append(0)

bars1 = ax.bar(x - width/2, n_means, width, yerr=n_sems, label='Normal',
               color=C_NORMAL, capsize=5, edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, a_means, width, yerr=a_sems, label='Anhedonic',
               color=C_ANHEDONIC, capsize=5, edgecolor='black', linewidth=0.5)

# Significance markers and % drop
for i, (p, pct) in enumerate(zip(p_vals, pct_drops)):
    if np.isnan(p):
        label = 'n/a'
        color = 'gray'
    else:
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
        label = f'{sig}\n({pct:.0f}%)' if p < 0.05 else 'n.s.'
        color = 'darkgreen' if p < 0.05 else 'gray'
    ymax = max(n_means[i] + n_sems[i], a_means[i] + a_sems[i])
    ax.text(i, ymax + 0.05, label, ha='center', fontsize=10, fontweight='bold', color=color)

ax.set_ylabel('Avg Positive Emotion Words', fontsize=13)
ax.set_title('Positive Emotion Words by Task Type', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(task_labels, fontsize=10)
ax.legend(fontsize=12, loc='upper left')
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

print('\nDetailed p-values:')
for tt, p, pct in zip(task_types, p_vals, pct_drops):
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.' if not np.isnan(p) else 'n/a'
    print(f'  {tt:30s} p={p:.4f}  {sig:5s}  ({pct:+.0f}%)')

## Block 11: Preference Ranking — Hedonic Valuation Gap

Does the anhedonic model show a reduced gap between how it ranks rewarding vs neutral activities?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: rank gap
ax = axes[0]
n_gap = df_summary['normal_rank_gap'].values
a_gap = df_summary['anhedonic_rank_gap'].values

bars = ax.bar(['Normal', 'Anhedonic'], [n_gap.mean(), a_gap.mean()],
              yerr=[n_gap.std()/np.sqrt(NUM_RUNS), a_gap.std()/np.sqrt(NUM_RUNS)],
              color=[C_NORMAL, C_ANHEDONIC], capsize=10, width=0.5,
              edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, [n_gap.mean(), a_gap.mean()]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')

for v in n_gap:
    ax.plot(0, v, 'o', color='black', alpha=0.5, markersize=5)
for v in a_gap:
    ax.plot(1, v, 'o', color='black', alpha=0.5, markersize=5)

gap_reduction = (1 - a_gap.mean() / n_gap.mean()) * 100
ax.set_ylabel('Rank Gap (neutral rank \u2212 reward rank)', fontsize=11)
ax.set_title(f'Hedonic Valuation Gap ({gap_reduction:.1f}% reduction)\np<0.0001 ***',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, max(n_gap.mean(), a_gap.mean()) * 1.35)

# Right: per-run paired comparison
ax = axes[1]
for i in range(NUM_RUNS):
    ax.plot([0, 1], [n_gap[i], a_gap[i]], 'o-', color='gray', alpha=0.6, markersize=8)
ax.plot([0, 1], [n_gap.mean(), a_gap.mean()], 's-', color='black',
        linewidth=3, markersize=12, zorder=5, label='Mean')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Normal', 'Anhedonic'])
ax.set_ylabel('Rank Gap', fontsize=11)
ax.set_title('Paired Run Comparison', fontsize=12, fontweight='bold')
ax.legend()

plt.suptitle('Preference Ranking Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Block 12: Reward vs Neutral — Hedonic Discrimination

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

n_vals = df_summary['normal_chose_reward_pct'].values
a_vals = df_summary['anhedonic_chose_reward_pct'].values

bars = ax.bar(['Normal', 'Anhedonic'], [n_vals.mean(), a_vals.mean()],
              yerr=[n_vals.std()/np.sqrt(NUM_RUNS), a_vals.std()/np.sqrt(NUM_RUNS)],
              color=[C_NORMAL, C_ANHEDONIC], capsize=10, width=0.5,
              edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, [n_vals.mean(), a_vals.mean()]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold')

for v in n_vals:
    ax.plot(0, v, 'o', color='black', alpha=0.5, markersize=5)
for v in a_vals:
    ax.plot(1, v, 'o', color='black', alpha=0.5, markersize=5)

ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Chance')
ax.set_ylabel('% Choosing Rewarding Option', fontsize=12)
ax.set_title('Hedonic Discrimination\np=0.0002 ***', fontsize=13, fontweight='bold')
ax.set_ylim(0, 105)
ax.legend()

plt.tight_layout()
plt.show()

drop = n_vals.mean() - a_vals.mean()
print(f'The anhedonic model chose the rewarding option {drop:.1f} percentage points less often.')

## Block 13: Scenario Continuation — Emotional Elaboration

In [ ]:
# Compute per-run stats for scenario continuation
cont_stats = []
for run_id in range(NUM_RUNS):
    rd = df_raw[(df_raw['run_id'] == run_id) & (df_raw['task_type'] == 'scenario_continuation')]
    cont_stats.append({
        'n_pos': rd['normal_emotion_pos'].mean(),
        'a_pos': rd['anhedonic_emotion_pos'].mean(),
        'n_flat': rd['normal_emotion_flat'].mean(),
        'a_flat': rd['anhedonic_emotion_flat'].mean(),
        'n_len': rd['normal_response_length'].mean(),
        'a_len': rd['anhedonic_response_length'].mean(),
    })
cs = pd.DataFrame(cont_stats)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

panels = [
    ('Positive Emotion Words', 'n_pos', 'a_pos', 'greater'),
    ('Flat / Neutral Words', 'n_flat', 'a_flat', 'less'),
    ('Response Length (words)', 'n_len', 'a_len', 'greater'),
]

for ax, (title, nc, ac, alt) in zip(axes, panels):
    nv = cs[nc].values
    av = cs[ac].values
    nm, am = nv.mean(), av.mean()
    ns, as_ = nv.std()/np.sqrt(NUM_RUNS), av.std()/np.sqrt(NUM_RUNS)

    bars = ax.bar(['Normal', 'Anhedonic'], [nm, am],
                  yerr=[ns, as_], color=[C_NORMAL, C_ANHEDONIC],
                  capsize=8, width=0.5, edgecolor='black', linewidth=0.5)
    ax.set_title(title, fontweight='bold')

    for v in nv:
        ax.plot(0, v, 'o', color='black', alpha=0.5, markersize=5)
    for v in av:
        ax.plot(1, v, 'o', color='black', alpha=0.5, markersize=5)

    try:
        t, p = stats.ttest_rel(nv, av, alternative=alt)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
        ymax = max(nm + ns, am + as_)
        ax.text(0.5, ymax * 1.08, f'p={p:.4f} {sig}', ha='center', fontsize=10,
                fontweight='bold', color='darkgreen' if p < 0.05 else 'gray')
    except:
        pass

plt.suptitle('Scenario Continuation: Emotional Elaboration', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

pct = (1 - cs['a_pos'].mean() / cs['n_pos'].mean()) * 100
print(f'Positive emotion words: {pct:.0f}% reduction')
print(f'Response length: {cs["n_len"].mean():.1f} \u2192 {cs["a_len"].mean():.1f} words ({(1-cs["a_len"].mean()/cs["n_len"].mean())*100:.0f}% shorter)')

## Block 14: Effort Willingness — Control Condition

Our probe targets **liking** (hedonic valuation). Effort willingness is driven by the **wanting** circuit. If our probe is specific, effort willingness should NOT change.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: % saying yes (the behavioral measure — should NOT change)
ax = axes[0]
n_e = df_summary['normal_effort_yes_pct'].values
a_e = df_summary['anhedonic_effort_yes_pct'].values

bars = ax.bar(['Normal', 'Anhedonic'], [n_e.mean(), a_e.mean()],
              yerr=[n_e.std()/np.sqrt(NUM_RUNS), a_e.std()/np.sqrt(NUM_RUNS)],
              color=[C_NORMAL, C_ANHEDONIC], capsize=10, width=0.5,
              edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, [n_e.mean(), a_e.mean()]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold')
ax.set_ylabel('% Willing to Exert Effort', fontsize=12)
ax.set_title('Effort Willingness (Behavioral)\np=1.0 n.s. \u2714', fontsize=12, fontweight='bold')
ax.set_ylim(0, 105)

# Right: emotion words in effort responses (the affect measure — DOES change)
ax = axes[1]
n_ep = df_summary['effort_willingness_normal_pos'].values
a_ep = df_summary['effort_willingness_anhedonic_pos'].values

bars = ax.bar(['Normal', 'Anhedonic'], [n_ep.mean(), a_ep.mean()],
              yerr=[n_ep.std()/np.sqrt(NUM_RUNS), a_ep.std()/np.sqrt(NUM_RUNS)],
              color=[C_NORMAL, C_ANHEDONIC], capsize=10, width=0.5,
              edgecolor='black', linewidth=0.5)

for v in n_ep:
    ax.plot(0, v, 'o', color='black', alpha=0.5, markersize=5)
for v in a_ep:
    ax.plot(1, v, 'o', color='black', alpha=0.5, markersize=5)

try:
    t, p = stats.ttest_rel(n_ep, a_ep, alternative='greater')
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
    ymax = max(n_ep.mean(), a_ep.mean()) * 1.15
    ax.text(0.5, ymax, f'p={p:.4f} {sig}', ha='center', fontsize=10, fontweight='bold',
            color='darkgreen' if p < 0.05 else 'gray')
except:
    pass

ax.set_ylabel('Avg Positive Emotion Words', fontsize=12)
ax.set_title('Emotion Words in Effort Responses\n(affect reduced, behavior unchanged)', fontsize=12, fontweight='bold')

plt.suptitle('Effort Willingness: Liking vs Wanting Dissociation', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print('KEY FINDING: The model still SAYS YES to effort at the same rate (wanting intact),')
print('but uses fewer positive emotion words when doing so (liking impaired).')
print('This is exactly the liking-wanting dissociation described in the clinical literature.')

## Block 15: Run-by-Run Consistency

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
runs = np.arange(NUM_RUNS) + 1

pairs = [
    ('Positive Emotion Words', 'normal_pos_words', 'anhedonic_pos_words'),
    ('Response Length', 'normal_resp_length', 'anhedonic_resp_length'),
    ('Reward Choice %', 'normal_chose_reward_pct', 'anhedonic_chose_reward_pct'),
]

for ax, (title, nc, ac) in zip(axes, pairs):
    nv = df_summary[nc].values
    av = df_summary[ac].values
    ax.plot(runs, nv, 'o-', color=C_NORMAL, label='Normal', linewidth=2, markersize=8)
    ax.plot(runs, av, 's-', color=C_ANHEDONIC, label='Anhedonic', linewidth=2, markersize=8)
    ax.fill_between(runs, nv, av, alpha=0.15, color='gray')
    ax.set_xlabel('Run')
    ax.set_title(title, fontweight='bold')
    ax.set_xticks(runs)
    ax.legend(fontsize=10)

plt.suptitle('Effect Consistency Across 5 Independent Runs', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Consistent gap in every run = reliable, reproducible effect.')

## Block 16: Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

def dash_bar(ax, title, n_col, a_col, ylabel, p_str, is_pct=False, from_raw_tt=None):
    """Helper to make a dashboard bar plot."""
    if from_raw_tt:
        # Compute per-run means from raw data for this task type
        nv = np.array([df_raw[(df_raw['run_id']==r) & (df_raw['task_type']==from_raw_tt)][n_col].mean() for r in range(NUM_RUNS)])
        av = np.array([df_raw[(df_raw['run_id']==r) & (df_raw['task_type']==from_raw_tt)][a_col].mean() for r in range(NUM_RUNS)])
    else:
        nv = df_summary[n_col].values
        av = df_summary[a_col].values
    
    nm, am = nv.mean(), av.mean()
    ns, as_ = nv.std()/np.sqrt(NUM_RUNS), av.std()/np.sqrt(NUM_RUNS)
    
    bars = ax.bar(['Normal', 'Anhedonic'], [nm, am], yerr=[ns, as_],
                  color=[C_NORMAL, C_ANHEDONIC], capsize=6, width=0.5,
                  edgecolor='black', linewidth=0.5)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(f'{title}\n{p_str}', fontsize=11, fontweight='bold')
    if is_pct:
        ax.set_ylim(0, 105)
    for v in nv:
        ax.plot(0, v, 'o', color='black', alpha=0.35, markersize=4)
    for v in av:
        ax.plot(1, v, 'o', color='black', alpha=0.35, markersize=4)

dash_bar(axes[0,0], 'Positive Emotion Words (Overall)',
         'normal_pos_words', 'anhedonic_pos_words', 'Avg Count', 'p=0.0001 ***')

dash_bar(axes[0,1], 'Preference Ranking Gap',
         'normal_rank_gap', 'anhedonic_rank_gap', 'Rank Gap', 'p<0.0001 ***')

dash_bar(axes[0,2], 'Reward vs Neutral Choice',
         'normal_chose_reward_pct', 'anhedonic_chose_reward_pct', '% Choosing Reward',
         'p=0.0002 ***', is_pct=True)

dash_bar(axes[1,0], 'Scenario Continuation',
         'scenario_continuation_normal_pos', 'scenario_continuation_anhedonic_pos',
         'Avg Pos Words', 'p=0.0001 ***')

dash_bar(axes[1,1], 'Anticipation',
         'anticipation_normal_pos', 'anticipation_anhedonic_pos',
         'Avg Pos Words', 'p<0.0001 ***')

dash_bar(axes[1,2], 'Effort Willingness (CONTROL)',
         'normal_effort_yes_pct', 'anhedonic_effort_yes_pct', '% Saying Yes',
         'p=1.0 n.s. \u2714', is_pct=True)

plt.suptitle('Anhedonia (Liking Probe) \u2014 Full Results Dashboard\n5 independent runs, mean \u00b1 SEM, paired t-test',
             fontsize=15, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig('dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print('Dashboard saved to dashboard.png')

## Block 17: Effect Size (Cohen's d)

In [ ]:
def cohens_d_paired(x, y):
    diff = x - y
    return diff.mean() / diff.std() if diff.std() > 0 else float('inf')

print('Effect Sizes (Cohen\'s d, paired):')
print('=' * 50)

effect_sizes = []
metrics = [
    ('Positive emotion words', 'normal_pos_words', 'anhedonic_pos_words'),
    ('Response length', 'normal_resp_length', 'anhedonic_resp_length'),
    ('Reward choice %', 'normal_chose_reward_pct', 'anhedonic_chose_reward_pct'),
]

for name, nc, ac in metrics:
    d = cohens_d_paired(df_summary[nc].values, df_summary[ac].values)
    size = 'huge' if abs(d) > 2 else 'large' if abs(d) > 0.8 else 'medium' if abs(d) > 0.5 else 'small'
    print(f'  {name:35s} d = {d:>6.2f}  ({size})')
    effect_sizes.append((name, d))

# Per task type
print('\nPer task type (positive emotion words):')
for tt in task_types:
    nc = f'{tt}_normal_pos'
    ac = f'{tt}_anhedonic_pos'
    nv = df_summary[nc].values
    av = df_summary[ac].values
    d = cohens_d_paired(nv, av)
    size = 'huge' if abs(d) > 2 else 'large' if abs(d) > 0.8 else 'medium' if abs(d) > 0.5 else 'small'
    print(f'  {tt:35s} d = {d:>6.2f}  ({size})')

## Block 18: Example Response Comparison

In [ ]:
run0 = df_raw[df_raw['run_id'] == 0]

for tt, n_examples in [('scenario_continuation', 3), ('anticipation', 3), ('reward_vs_neutral', 2)]:
    subset = run0[run0['task_type'] == tt].head(n_examples)
    print(f'\n{"=" * 70}')
    print(f'  {tt.upper().replace("_", " ")}')
    print(f'{"=" * 70}')
    for _, row in subset.iterrows():
        print(f'\n  NORMAL:    {row["normal_response"][:300]}')
        print(f'  ANHEDONIC: {row["anhedonic_response"][:300]}')
        print(f'  [pos: {int(row["normal_emotion_pos"])} \u2192 {int(row["anhedonic_emotion_pos"])}  |  '
              f'flat: {int(row["normal_emotion_flat"])} \u2192 {int(row["anhedonic_emotion_flat"])}]')
        print(f'  {"-" * 60}')

---
## Summary & Interpretation

### Multi-Run Results (5 independent runs, paired t-test)

| Metric | Normal | Anhedonic | p-value | Effect |
|--------|--------|-----------|---------|--------|
| **Positive emotion words** | 0.752 \u00b1 0.012 | 0.645 \u00b1 0.016 | **p=0.0001 \*\*\*** | 14% reduction |
| **Response length** | 64.9 \u00b1 0.3 | 62.1 \u00b1 0.3 | **p<0.0001 \*\*\*** | 4% shorter |
| **Preference ranking gap** | 1.831 | 1.694 | **p<0.0001 \*\*\*** | 7.5% reduction |
| **Reward vs neutral choice** | 75.0% | 68.5% \u00b1 1.4% | **p=0.0002 \*\*\*** | 6.5pp drop |
| **Scenario continuation** | 0.500 | 0.355 \u00b1 0.021 | **p=0.0001 \*\*\*** | 29% reduction |
| **Anticipation** | 1.260 \u00b1 0.014 | 1.150 | **p<0.0001 \*\*\*** | 8.7% reduction |
| **Effort willingness** | 77.5% | 80.0% | p=1.0 n.s. | **Control** \u2714 |

### Key Finding: Liking-Wanting Dissociation

The intervention selectively disrupts **consummatory anhedonia (liking)**:
- Positive emotion words reduced across all affective tasks
- Hedonic valuation and discrimination weakened
- Emotional elaboration shorter and flatter

While **motivational drive (wanting) is preserved**:
- Effort willingness: unchanged at 77.5-80%
- The model still says YES to effort, but with **less enthusiasm** (emotion words drop)

This mirrors the clinical dissociation between liking and wanting (Berridge & Robinson, 2003).

### Next Steps
1. Build separate probes for **wanting**, **enjoying**, and **learning**
2. Create permanent anhedonic model via weight surgery
3. Combine probes for full 4-component anhedonia model